In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\HP\anaconda3\envs\spark_env\python.exe
Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]


## 1. Create the Spark session


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EcommerceSessionDataPreparation")
    .master("local[2]")
    .config("spark.driver.memory", "10g")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.executor.memory", "4g")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
spark.range(5).show()


Spark version: 3.5.8
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



## 2. Configure HDFS paths and batch size


In [3]:
HDFS_SESSION_ROOT = "hdfs://localhost:9000/ecommerce/raw/sessions"
OUTPUT_PATH = "hdfs://localhost:9000/ecommerce/features/session_conversion"

TOTAL_FILES = 20
FILES_PER_BATCH = 4
OUTPUT_PARTITIONS = 8

print("Input:", HDFS_SESSION_ROOT)
print("Output:", OUTPUT_PATH)
print("Total files:", TOTAL_FILES)
print("Files per batch:", FILES_PER_BATCH)


Input: hdfs://localhost:9000/ecommerce/raw/sessions
Output: hdfs://localhost:9000/ecommerce/features/session_conversion
Total files: 20
Files per batch: 4


## 3. Define the session schema

The explicit schema is essential because `cart_contents` contains dynamic product IDs. Without this schema, Spark may infer it as a very large struct instead of a map.


In [4]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    ArrayType,
    MapType
)

geo_schema = StructType([
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("ip_address", StringType(), True)
])

device_schema = StructType([
    StructField("type", StringType(), True),
    StructField("os", StringType(), True),
    StructField("browser", StringType(), True)
])

page_view_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("page_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("view_duration", LongType(), True)
])

cart_item_schema = StructType([
    StructField("quantity", LongType(), True),
    StructField("price", DoubleType(), True)
])

session_schema = StructType([
    StructField("session_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("duration_seconds", LongType(), True),
    StructField("geo_data", geo_schema, True),
    StructField("device_profile", device_schema, True),
    StructField("viewed_products", ArrayType(StringType()), True),
    StructField("page_views", ArrayType(page_view_schema), True),
    StructField(
        "cart_contents",
        MapType(StringType(), cart_item_schema),
        True
    ),
    StructField("conversion_status", StringType(), True),
    StructField("referrer", StringType(), True)
])

print(session_schema.simpleString())


struct<session_id:string,user_id:string,start_time:string,end_time:string,duration_seconds:bigint,geo_data:struct<city:string,state:string,country:string,ip_address:string>,device_profile:struct<type:string,os:string,browser:string>,viewed_products:array<string>,page_views:array<struct<timestamp:string,page_type:string,product_id:string,category_id:string,view_duration:bigint>>,cart_contents:map<string,struct<quantity:bigint,price:double>>,conversion_status:string,referrer:string>


## 4. Define the cleaning and feature-engineering function


In [5]:
from pyspark.sql.functions import (
    col,
    size,
    to_timestamp,
    hour,
    dayofweek,
    when,
    lit,
    coalesce
)

def prepare_session_batch(df):
    return (
        df
        .filter(col("session_id").isNotNull())
        .dropDuplicates(["session_id"])
        .withColumn("session_start", to_timestamp("start_time"))
        .withColumn("session_end", to_timestamp("end_time"))
        .filter(
            col("duration_seconds").isNotNull()
            & (col("duration_seconds") >= 0)
        )
        .withColumn(
            "viewed_product_count",
            coalesce(size("viewed_products"), lit(0))
        )
        .withColumn(
            "page_view_count",
            coalesce(size("page_views"), lit(0))
        )
        .withColumn(
            "cart_product_count",
            coalesce(size("cart_contents"), lit(0))
        )
        .withColumn("session_hour", hour("session_start"))
        .withColumn(
            "session_day_of_week",
            dayofweek("session_start")
        )
        .withColumn("device_type", col("device_profile.type"))
        .withColumn("operating_system", col("device_profile.os"))
        .withColumn("browser", col("device_profile.browser"))
        .withColumn("country", col("geo_data.country"))
        .withColumn(
            "label",
            when(
                col("conversion_status") == "converted",
                lit(1.0)
            ).otherwise(lit(0.0))
        )
        .select(
            "session_id",
            "user_id",
            "session_start",
            "session_end",
            "duration_seconds",
            "viewed_product_count",
            "page_view_count",
            "cart_product_count",
            "session_hour",
            "session_day_of_week",
            "device_type",
            "operating_system",
            "browser",
            "country",
            "referrer",
            "conversion_status",
            "label"
        )
    )


## 5. Test one file before processing all batches


## 6. Process all session files in batches

Each batch is persisted to disk only, written to Parquet, then removed from Spark storage before the next batch.


In [6]:
import gc
from pyspark import StorageLevel

first_batch = True
total_processed = 0

for start_index in range(0, TOTAL_FILES, FILES_PER_BATCH):
    end_index = min(
        start_index + FILES_PER_BATCH,
        TOTAL_FILES
    )

    batch_paths = [
        f"{HDFS_SESSION_ROOT}/sessions_{i}.json"
        for i in range(start_index, end_index)
    ]

    print(
        f"\nProcessing sessions_{start_index}.json "
        f"to sessions_{end_index - 1}.json"
    )

    raw_batch = (
        spark.read
        .schema(session_schema)
        .option("multiLine", "true")
        .json(batch_paths)
    )

    prepared_batch = (
        prepare_session_batch(raw_batch)
        .repartition(OUTPUT_PARTITIONS)
        .persist(StorageLevel.DISK_ONLY)
    )

    try:
        batch_count = prepared_batch.count()
        total_processed += batch_count

        print(f"Prepared records: {batch_count:,}")

        write_mode = "overwrite" if first_batch else "append"

        (
            prepared_batch.write
            .mode(write_mode)
            .partitionBy("label")
            .parquet(OUTPUT_PATH)
        )

        print("Batch saved successfully.")

    finally:
        prepared_batch.unpersist()
        del raw_batch
        del prepared_batch
        spark.catalog.clearCache()
        gc.collect()

    first_batch = False

print(f"\nAll batches completed. Total records processed: {total_processed:,}")



Processing sessions_0.json to sessions_3.json
Prepared records: 399,999
Batch saved successfully.

Processing sessions_4.json to sessions_7.json
Prepared records: 400,000
Batch saved successfully.

Processing sessions_8.json to sessions_11.json
Prepared records: 400,000
Batch saved successfully.

Processing sessions_12.json to sessions_15.json
Prepared records: 400,000
Batch saved successfully.

Processing sessions_16.json to sessions_19.json
Prepared records: 400,000
Batch saved successfully.

All batches completed. Total records processed: 1,999,999


## 7. Read and validate the prepared Parquet dataset



In [8]:
session_df = spark.read.parquet(OUTPUT_PATH)

prepared_count = session_df.count()

print("Prepared sessions:", f"{prepared_count:,}")
session_df.printSchema()

session_df.groupBy(
    "label",
    "conversion_status"
).count().orderBy("label", "conversion_status").show()


Prepared sessions: 1,999,999
root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- session_end: timestamp (nullable = true)
 |-- duration_seconds: long (nullable = true)
 |-- viewed_product_count: integer (nullable = true)
 |-- page_view_count: integer (nullable = true)
 |-- cart_product_count: integer (nullable = true)
 |-- session_hour: integer (nullable = true)
 |-- session_day_of_week: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- operating_system: string (nullable = true)
 |-- browser: string (nullable = true)
 |-- country: string (nullable = true)
 |-- referrer: string (nullable = true)
 |-- conversion_status: string (nullable = true)
 |-- label: double (nullable = true)

+-----+-----------------+------+
|label|conversion_status| count|
+-----+-----------------+------+
|  0.0|        abandoned|892802|
|  0.0|          browsed|701520|
|  1.0|        converted|405677|
+-

## 8. Inspect a small sample


In [9]:
session_df.select(
    "session_id",
    "user_id",
    "duration_seconds",
    "viewed_product_count",
    "page_view_count",
    "cart_product_count",
    "device_type",
    "conversion_status",
    "label"
).show(20, truncate=False)


+---------------+-----------+----------------+--------------------+---------------+------------------+-----------+-----------------+-----+
|session_id     |user_id    |duration_seconds|viewed_product_count|page_view_count|cart_product_count|device_type|conversion_status|label|
+---------------+-----------+----------------+--------------------+---------------+------------------+-----------+-----------------+-----+
|sess_3cd5d12b6c|user_002640|3381            |2                   |9              |1                 |desktop    |abandoned        |0.0  |
|sess_053c14ee56|user_000440|2893            |1                   |4              |0                 |tablet     |browsed          |0.0  |
|sess_db696a2c27|user_005874|2004            |5                   |16             |2                 |desktop    |abandoned        |0.0  |
|sess_b74120bde2|user_001946|1146            |5                   |13             |1                 |mobile     |abandoned        |0.0  |
|sess_ff3c0c0905|user_00226

In [10]:
HDFS_ROOT = "hdfs://localhost:9000/ecommerce/raw"

USERS_PATH = f"{HDFS_ROOT}/users/users.json"
PRODUCTS_PATH = f"{HDFS_ROOT}/products/products.json"
CATEGORIES_PATH = f"{HDFS_ROOT}/categories/categories.json"
TRANSACTIONS_PATH = f"{HDFS_ROOT}/transactions/transactions.json"

CURATED_ROOT = "hdfs://localhost:9000/ecommerce/curated"

In [11]:
users_raw = (
    spark.read
    .option("multiLine", "true")
    .json(USERS_PATH)
)

products_raw = (
    spark.read
    .option("multiLine", "true")
    .json(PRODUCTS_PATH)
)

categories_raw = (
    spark.read
    .option("multiLine", "true")
    .json(CATEGORIES_PATH)
)

transactions_raw = (
    spark.read
    .option("multiLine", "true")
    .json(TRANSACTIONS_PATH)
)

In [12]:
users_raw.printSchema()
products_raw.printSchema()
categories_raw.printSchema()
transactions_raw.printSchema()

root
 |-- geo_data: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- last_active: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- user_id: string (nullable = true)

root
 |-- base_price: double (nullable = true)
 |-- category_id: string (nullable = true)
 |-- creation_date: string (nullable = true)
 |-- current_stock: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- name: string (nullable = true)
 |-- price_history: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- subcategory_id: string (nullable = true)

root
 |-- category_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- subcategories: array (nullable = true)
 |    |-- element: struct (containsNul

In [13]:
# Clean The Users
from pyspark.sql.functions import col, to_timestamp

users_clean = (
    users_raw
    .filter(col("user_id").isNotNull())
    .dropDuplicates(["user_id"])
    .withColumn(
        "registration_timestamp",
        to_timestamp("registration_date")
    )
    .withColumn(
        "last_active_timestamp",
        to_timestamp("last_active")
    )
    .select(
        "user_id",
        col("geo_data.city").alias("city"),
        col("geo_data.state").alias("state"),
        col("geo_data.country").alias("country"),
        "registration_timestamp",
        "last_active_timestamp"
    )
)

In [14]:
# Clean Products
products_clean = (
    products_raw
    .filter(col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
    .filter(col("base_price") >= 0)
    .filter(col("current_stock") >= 0)
    .withColumn(
        "creation_timestamp",
        to_timestamp("creation_date")
    )
    .select(
        "product_id",
        col("name").alias("product_name"),
        "category_id",
        "subcategory_id",
        "base_price",
        "current_stock",
        "is_active",
        "price_history",
        "creation_timestamp"
    )
)

In [15]:
# Clean Categories
categories_clean = (
    categories_raw
    .filter(col("category_id").isNotNull())
    .dropDuplicates(["category_id"])
    .select(
        "category_id",
        col("name").alias("category_name"),
        "subcategories"
    )
)

In [16]:
# Clean Transactions
transactions_clean = (
    transactions_raw
    .filter(col("transaction_id").isNotNull())
    .dropDuplicates(["transaction_id"])
    .filter(col("subtotal") >= 0)
    .filter(col("discount") >= 0)
    .filter(col("total") >= 0)
    .withColumn(
        "transaction_timestamp",
        to_timestamp("timestamp")
    )
    .select(
        "transaction_id",
        "session_id",
        "user_id",
        "transaction_timestamp",
        "items",
        "subtotal",
        "discount",
        "total",
        "payment_method",
        "status"
    )
)

In [18]:
# Validate Cleaned Data
users_clean.show(5, truncate=False)
products_clean.show(5, truncate=False)
categories_clean.show(5, truncate=False)
transactions_clean.show(5, truncate=False)

+-----------+------------------+-----+-------+----------------------+---------------------+
|user_id    |city              |state|country|registration_timestamp|last_active_timestamp|
+-----------+------------------+-----+-------+----------------------+---------------------+
|user_000000|Lake Wendy        |MH   |KG     |2025-10-20 04:39:42   |2026-05-18 15:55:38  |
|user_000001|New Kevin         |HI   |LB     |2025-12-19 07:36:46   |2026-05-23 16:49:26  |
|user_000002|Morseport         |MO   |CU     |2025-10-26 19:33:27   |2026-05-24 14:30:46  |
|user_000003|Matthewmouth      |GU   |PA     |2025-09-21 18:44:45   |2026-01-07 05:46:43  |
|user_000004|South Michaelmouth|MI   |BR     |2026-01-27 09:46:35   |2026-04-08 03:10:27  |
+-----------+------------------+-----+-------+----------------------+---------------------+
only showing top 5 rows

+----------+-------------------------------------+-----------+--------------+----------+-------------+---------+-----------------------------------

In [20]:
# Save as Parquet
users_clean.write \
    .mode("overwrite") \
    .parquet(f"{CURATED_ROOT}/users")

products_clean.write \
    .mode("overwrite") \
    .parquet(f"{CURATED_ROOT}/products")

categories_clean.write \
    .mode("overwrite") \
    .parquet(f"{CURATED_ROOT}/categories")

transactions_clean.repartition(8).write \
    .mode("overwrite") \
    .parquet(f"{CURATED_ROOT}/transactions")

In [21]:
# Reload Optimized Datasets
users_df = spark.read.parquet(
    f"{CURATED_ROOT}/users"
)

products_df = spark.read.parquet(
    f"{CURATED_ROOT}/products"
)

categories_df = spark.read.parquet(
    f"{CURATED_ROOT}/categories"
)

transactions_df = spark.read.parquet(
    f"{CURATED_ROOT}/transactions"
)

In [22]:
print("Users:", users_df.count())
print("Products:", products_df.count())
print("Categories:", categories_df.count())
print("Transactions:", transactions_df.count())

Users: 10000
Products: 5000
Categories: 25
Transactions: 500000


## RECOMMENDATION- SYSTEM PIPELINE

1. Validate schemas and types
2. Remove irrelevant and invalid records
3. Normalize transaction items
4. Join products and categories
5. Create user-product interactions
6. Remove extreme values
7. Engineer product and user features
8. Analyze correlations
9. Leave-one-transaction-out split
10. Train ALS collaborative model
11. Build content-similarity model
12. Blend both recommendation scores
13. Remove already-purchased products
14. Evaluate Recall@K, Precision@K and MAP@K
15. Save recommendations

In [24]:
print("Users:", users_df.count())
print("Products:", products_df.count())
print("Categories:", categories_df.count())
print("Transactions:", transactions_df.count())
print("Session features:", session_df.count())

Users: 10000
Products: 5000
Categories: 25
Transactions: 500000
Session features: 1999999


In [26]:
# Inspecting Schemas
users_df.printSchema()
products_df.printSchema()
categories_df.printSchema()
transactions_df.printSchema()
session_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_timestamp: timestamp (nullable = true)
 |-- last_active_timestamp: timestamp (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- subcategory_id: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- current_stock: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- price_history: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- creation_timestamp: timestamp (nullable = true)

root
 |-- category_id: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- subcategories: array (nullable = true)
 |    |-- element: struct (containsNull = true)

In [27]:
# Normalize Transaction Line Items
from pyspark.sql.functions import (
    col,
    explode,
    sum as spark_sum,
    countDistinct,
    max as spark_max,
    min as spark_min,
    avg,
    log1p,
    when,
    lit,
    row_number,
    count
)

transaction_items = (
    transactions_df
    .filter(
        col("status").isin(
            "completed",
            "shipped",
            "delivered"
        )
    )
    .filter(col("user_id").isNotNull())
    .filter(col("transaction_id").isNotNull())
    .filter(col("transaction_timestamp").isNotNull())
    .withColumn("item", explode("items"))
    .select(
        "transaction_id",
        "user_id",
        "transaction_timestamp",
        "status",
        col("item.product_id").alias("product_id"),
        col("item.quantity").cast("double").alias("quantity"),
        col("item.unit_price").cast("double").alias("unit_price"),
        col("item.subtotal").cast("double").alias("line_subtotal")
    )
    .filter(col("product_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("unit_price") >= 0)
    .filter(col("line_subtotal") >= 0)
)

In [28]:
transaction_items.show(10, truncate=False)
transaction_items.printSchema()

+----------------+-----------+---------------------+---------+----------+--------+----------+-------------+
|transaction_id  |user_id    |transaction_timestamp|status   |product_id|quantity|unit_price|line_subtotal|
+----------------+-----------+---------------------+---------+----------+--------+----------+-------------+
|txn_24def3f37134|user_006383|2026-06-02 16:33:32  |completed|prod_03774|1.0     |88.36     |88.36        |
|txn_24def3f37134|user_006383|2026-06-02 16:33:32  |completed|prod_04787|2.0     |7.43      |14.86        |
|txn_24def3f37134|user_006383|2026-06-02 16:33:32  |completed|prod_02448|2.0     |385.02    |770.04       |
|txn_e94c7ed514ef|user_002709|2026-03-14 16:41:47  |completed|prod_02976|2.0     |212.48    |424.96       |
|txn_9e1c3a899238|user_009057|2026-04-06 10:39:35  |completed|prod_01688|2.0     |359.0     |718.0        |
|txn_eddbe2041c55|user_002519|2026-05-04 19:03:25  |completed|prod_01818|3.0     |149.19    |447.57       |
|txn_eddbe2041c55|user_00251

In [29]:
# Remove Irrelevant Products
valid_products = (
    products_df
    .filter(col("product_id").isNotNull())
    .filter(col("is_active") == True)
    .filter(col("current_stock") > 0)
    .filter(col("base_price") > 0)
    .dropDuplicates(["product_id"])
)

In [30]:
# Join Valid Products
transaction_items = (
    transaction_items
    .join(
        valid_products.select("product_id"),
        "product_id",
        "inner"
    )
)

In [31]:
# Removing Extreme Quantity and Price Values
# Use Percentile Based Capping Instead Of Arbitray Thresholds
quantity_limits = transaction_items.approxQuantile(
    "quantity",
    [0.01, 0.99],
    0.01
)

price_limits = transaction_items.approxQuantile(
    "unit_price",
    [0.01, 0.99],
    0.01
)

quantity_lower, quantity_upper = quantity_limits
price_lower, price_upper = price_limits

print("Quantity limits:", quantity_limits)
print("Price limits:", price_limits)

Quantity limits: [1.0, 6.0]
Price limits: [4.6, 594.15]


In [32]:
# Filter
transaction_items_clean = (
    transaction_items
    .filter(
        col("quantity").between(
            quantity_lower,
            quantity_upper
        )
    )
    .filter(
        col("unit_price").between(
            price_lower,
            price_upper
        )
    )
)

In [33]:
# Create a Central Product Dimension
product_dimension = (
    valid_products
    .select(
        "product_id",
        "product_name",
        "category_id",
        "subcategory_id",
        "base_price",
        "current_stock",
        "is_active"
    )
    .join(
        categories_df.select(
            "category_id",
            "category_name"
        ),
        "category_id",
        "left"
    )
)

In [34]:
# Check
product_dimension.show(10, truncate=False)

+-----------+----------+-------------------------------------+--------------+----------+-------------+---------+--------------------------+
|category_id|product_id|product_name                         |subcategory_id|base_price|current_stock|is_active|category_name             |
+-----------+----------+-------------------------------------+--------------+----------+-------------+---------+--------------------------+
|cat_016    |prod_00000|Fully-Configurable Static Benchmark  |sub_016_00    |243.53    |367          |true     |Wright-Leblanc            |
|cat_009    |prod_00001|Open-Source Regional Synergy         |sub_009_00    |28.09     |6            |true     |Skinner-Romero            |
|cat_024    |prod_00002|Front-Line Cohesive Website          |sub_024_04    |384.09    |119          |true     |Patterson, Smith and Jones|
|cat_017    |prod_00003|Assimilated Context-Sensitive Support|sub_017_01    |120.5     |18           |true     |Hancock Inc               |
|cat_022    |prod_00

In [35]:
# Create Transaction-Level Test Split
# We use the latest transaction for testing, but only for users who have at least two transactions. 
# Therefore, every test user remains represented in training.

from pyspark.sql.window import Window

eligible_users = (
    transaction_items_clean
    .select(
        "user_id",
        "transaction_id"
    )
    .dropDuplicates()
    .groupBy("user_id")
    .agg(
        countDistinct("transaction_id").alias(
            "transaction_count"
        )
    )
    .filter(col("transaction_count") >= 2)
)

In [36]:
# Restrict only to eligible users
eligible_items = (
    transaction_items_clean
    .join(
        eligible_users.select("user_id"),
        "user_id",
        "inner"
    )
)

In [37]:
# Rank The Transaction from the newest to oldest
transaction_dates = (
    eligible_items
    .select(
        "user_id",
        "transaction_id",
        "transaction_timestamp"
    )
    .dropDuplicates()
)

latest_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        col("transaction_timestamp").desc(),
        col("transaction_id").desc()
    )
)

ranked_transactions = (
    transaction_dates
    .withColumn(
        "transaction_rank",
        row_number().over(latest_window)
    )
)

In [38]:
# Create the latest Test Transactions
test_transaction_ids = (
    ranked_transactions
    .filter(col("transaction_rank") == 1)
    .select(
        "user_id",
        "transaction_id"
    )
)

In [39]:
# Creating the Training Interactions
train_items = (
    eligible_items.alias("i")
    .join(
        test_transaction_ids.alias("t"),
        (
            col("i.user_id") == col("t.user_id")
        ) & (
            col("i.transaction_id") ==
            col("t.transaction_id")
        ),
        "left_anti"
    )
)

In [40]:
# Creating The Test Items
test_items = (
    eligible_items.alias("i")
    .join(
        test_transaction_ids.alias("t"),
        (
            col("i.user_id") == col("t.user_id")
        ) & (
            col("i.transaction_id") ==
            col("t.transaction_id")
        ),
        "inner"
    )
    .select(
        col("i.user_id").alias("user_id"),
        col("i.transaction_id").alias("transaction_id"),
        col("i.product_id").alias("product_id"),
        col("i.transaction_timestamp").alias(
            "transaction_timestamp"
        )
    )
)

In [41]:
# Verify that every test user exist in training
test_users_without_training = (
    test_items.select("user_id").distinct()
    .join(
        train_items.select("user_id").distinct(),
        "user_id",
        "left_anti"
    )
)

print(
    "Test users missing from training:",
    test_users_without_training.count()
)

Test users missing from training: 0


## 8. Aggregate Implicit User-Product Interactions

ALS needs numeric user IDs, numeric product IDs and a rating or Interaction Score
We will use implicit feedback

In [57]:
user_product_interactions = (
    train_items
    .groupBy(
        "user_id",
        "product_id"
    )
    .agg(
        spark_sum("quantity").alias("total_quantity"),
        spark_sum("line_subtotal").alias("total_spending"),
        countDistinct("transaction_id").alias(
            "purchase_frequency"
        ),
        spark_max("transaction_timestamp").alias(
            "last_purchase_timestamp"
        )
    )
)

In [58]:
# Create a stable Interaction Strenght
# Avoiding very expensive purchases to dominate the model
user_product_interactions = (
    user_product_interactions
    .withColumn(
        "interaction_strength",
        log1p(col("total_quantity")) +
        1.5 * log1p(col("purchase_frequency")) +
        0.25 * log1p(col("total_spending"))
    )
)

In [59]:
# Create Integer User And Product Mappings
# MLlib ALS requires Integer Identifiers
user_window = Window.orderBy("user_id")
product_window = Window.orderBy("product_id")

user_mapping = (
    user_product_interactions
    .select("user_id")
    .distinct()
    .withColumn(
        "user_index",
        (
            row_number().over(user_window) - 1
        ).cast("integer")
    )
)

product_mapping = (
    product_dimension
    .select("product_id")
    .distinct()
    .withColumn(
        "product_index",
        (
            row_number().over(product_window) - 1
        ).cast("integer")
    )
)

In [60]:
# Map Interactions
als_training_data = (
    user_product_interactions
    .join(user_mapping, "user_id", "inner")
    .join(product_mapping, "product_id", "inner")
    .select(
        "user_id",
        "product_id",
        "user_index",
        "product_index",
        col("interaction_strength").cast("float").alias("rating")
    )
)

In [61]:
als_training_data.show(10, truncate=False)

+-----------+----------+----------+-------------+---------+
|user_id    |product_id|user_index|product_index|rating   |
+-----------+----------+----------+-------------+---------+
|user_002564|prod_03408|2564      |1531         |3.8119798|
|user_004710|prod_02008|4710      |890          |2.8427353|
|user_001931|prod_04420|1931      |1966         |3.8555658|
|user_003461|prod_01810|3461      |808          |3.4351058|
|user_009687|prod_04515|9687      |2017         |5.096739 |
|user_000832|prod_02445|832       |1084         |3.6630313|
|user_004907|prod_00064|4907      |35           |3.623533 |
|user_007729|prod_00741|7729      |332          |4.8905563|
|user_007236|prod_00121|7236      |63           |4.1695504|
|user_008640|prod_00310|8640      |138          |4.141655 |
+-----------+----------+----------+-------------+---------+
only showing top 10 rows



# Correlation Analysis
Correlation is useful for numeric behaviour features, 
but it does not directly determine the final recommender features because recommendations also depend on categorical product attributes.


In [63]:
# User-Product Numeric Features
correlation_data = (
    user_product_interactions
    .select(
        "total_quantity",
        "total_spending",
        "purchase_frequency",
        "interaction_strength"
    )
    .dropna()
)

In [64]:
# MLlib Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

correlation_columns = [
    "total_quantity",
    "total_spending",
    "purchase_frequency",
    "interaction_strength"
]

correlation_assembler = VectorAssembler(
    inputCols=correlation_columns,
    outputCol="correlation_features"
)

correlation_vectors = (
    correlation_assembler
    .transform(correlation_data)
    .select("correlation_features")
)

correlation_matrix = (
    Correlation.corr(
        correlation_vectors,
        "correlation_features",
        method="pearson"
    )
    .head()[0]
    .toArray()
)

print(correlation_columns)
print(correlation_matrix)

['total_quantity', 'total_spending', 'purchase_frequency', 'interaction_strength']
[[1.         0.55593777 0.25217997 0.87480626]
 [0.55593777 1.         0.13999258 0.80887283]
 [0.25217997 0.13999258 1.         0.29327836]
 [0.87480626 0.80887283 0.29327836 1.        ]]


In [65]:
# Train Collaborative Filtering with ALS
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_index",
    itemCol="product_index",
    ratingCol="rating",
    implicitPrefs=True,
    rank=50,
    maxIter=15,
    regParam=0.08,
    alpha=20.0,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=42
)

als_model = als.fit(als_training_data)

In [67]:
CURATED_ROOT = "hdfs://localhost:9000/ecommerce/curated"
FEATURE_ROOT = "hdfs://localhost:9000/ecommerce/features"
MODEL_ROOT = "hdfs://localhost:9000/ecommerce/models"
ANALYTICS_ROOT = "hdfs://localhost:9000/ecommerce/analytics"

In [68]:
als_model.write().overwrite().save(
    f"{MODEL_ROOT}/hybrid_recommender_als"
)

In [69]:
# Generate more candidates than needed because purchased products will be removed
als_candidates_raw = (
    als_model.recommendForAllUsers(100)
)

In [70]:
als_candidates = (
    als_candidates_raw
    .withColumn(
        "recommendation",
        explode("recommendations")
    )
    .select(
        "user_index",
        col("recommendation.product_index").alias(
            "product_index"
        ),
        col("recommendation.rating").alias(
            "als_score"
        )
    )
    .join(
        user_mapping,
        "user_index",
        "inner"
    )
    .join(
        product_mapping,
        "product_index",
        "inner"
    )
)

## Build Content-Based Product Features

In [71]:
# Start by Indexing Categorical Attributes
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    StandardScaler
)
from pyspark.ml import Pipeline

content_products = (
    product_dimension
    .fillna({
        "category_id": "unknown_category",
        "subcategory_id": "unknown_subcategory",
        "base_price": 0.0
    })
)

In [72]:
# Create Price Features
price_stats = content_products.agg(
    avg("base_price").alias("mean_price")
).first()

mean_price = float(price_stats["mean_price"])

In [73]:
# Build Pipeline
category_indexer = StringIndexer(
    inputCol="category_id",
    outputCol="category_index",
    handleInvalid="keep"
)

subcategory_indexer = StringIndexer(
    inputCol="subcategory_id",
    outputCol="subcategory_index",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=[
        "category_index",
        "subcategory_index"
    ],
    outputCols=[
        "category_vector",
        "subcategory_vector"
    ]
)

content_assembler = VectorAssembler(
    inputCols=[
        "category_vector",
        "subcategory_vector",
        "base_price"
    ],
    outputCol="raw_content_features"
)

content_scaler = StandardScaler(
    inputCol="raw_content_features",
    outputCol="content_features",
    withMean=False,
    withStd=True
)

content_pipeline = Pipeline(
    stages=[
        category_indexer,
        subcategory_indexer,
        encoder,
        content_assembler,
        content_scaler
    ]
)

content_model = content_pipeline.fit(content_products)

product_content_vectors = (
    content_model
    .transform(content_products)
    .select(
        "product_id",
        "product_name",
        "category_id",
        "subcategory_id",
        "base_price",
        "content_features"
    )
)

## Create User Content Profiles


A user profile is the weighted average of product features from products purchased in training.

Spark does not have a built-in sparse-vector weighted-average aggregator, so for 5,000 products it is reasonable to compute content similarity through category, subcategory and price affinity rather than creating a massive product-to-product Cartesian similarity matrix.

In [74]:
# User Category Preference
user_category_preferences = (
    train_items
    .join(
        product_dimension.select(
            "product_id",
            "category_id",
            "subcategory_id",
            "base_price"
        ),
        "product_id",
        "inner"
    )
    .groupBy(
        "user_id",
        "category_id",
        "subcategory_id"
    )
    .agg(
        spark_sum("quantity").alias("preference_weight"),
        avg("base_price").alias("preferred_price")
    )
)

In [75]:
# Create Total User Weights
user_total_preferences = (
    user_category_preferences
    .groupBy("user_id")
    .agg(
        spark_sum("preference_weight").alias(
            "total_preference_weight"
        )
    )
)

In [76]:
# Normalize 
user_category_preferences = (
    user_category_preferences
    .join(
        user_total_preferences,
        "user_id",
        "inner"
    )
    .withColumn(
        "category_affinity",
        col("preference_weight") /
        col("total_preference_weight")
    )
)

## Create Content-Based Candidate Scores

Join each user’s preferred categories with valid products in the same categories.

In [78]:
from pyspark.sql.functions import abs, col
content_candidates = (
    user_category_preferences.alias("u")
    .join(
        product_dimension.alias("p"),
        (
            col("u.category_id") ==
            col("p.category_id")
        ),
        "inner"
    )
    .withColumn(
        "subcategory_match",
        when(
            col("u.subcategory_id") ==
            col("p.subcategory_id"),
            lit(1.0)
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "price_similarity",
        lit(1.0) /
        (
            lit(1.0) +
            (
                abs(
                    col("p.base_price") -
                    col("u.preferred_price")
                ) /
                (
                    col("u.preferred_price") +
                    lit(1.0)
                )
            )
        )
    )
    .withColumn(
        "content_score",
        0.55 * col("category_affinity") +
        0.25 * col("subcategory_match") +
        0.20 * col("price_similarity")
    )
    .select(
        col("u.user_id").alias("user_id"),
        col("p.product_id").alias("product_id"),
        "content_score"
    )
)

In [79]:
# Remove Products Already Purchased
purchased_products = (
    train_items
    .select(
        "user_id",
        "product_id"
    )
    .distinct()
)

In [80]:
# Filter ALS
als_candidates_filtered = (
    als_candidates.alias("r")
    .join(
        purchased_products.alias("p"),
        (
            col("r.user_id") == col("p.user_id")
        ) & (
            col("r.product_id") ==
            col("p.product_id")
        ),
        "left_anti"
    )
)

In [81]:
# Filter Content Candidates
content_candidates_filtered = (
    content_candidates.alias("r")
    .join(
        purchased_products.alias("p"),
        (
            col("r.user_id") == col("p.user_id")
        ) & (
            col("r.product_id") ==
            col("p.product_id")
        ),
        "left_anti"
    )
)

## Normalize ALS and Content Scores

Scores from the two models are on different scales.

In [82]:
from pyspark.sql.functions import max as spark_max, min as spark_min

als_score_limits = als_candidates_filtered.agg(
    spark_min("als_score").alias("minimum"),
    spark_max("als_score").alias("maximum")
).first()

als_min = float(als_score_limits["minimum"])
als_max = float(als_score_limits["maximum"])

als_candidates_normalized = (
    als_candidates_filtered
    .withColumn(
        "als_score_normalized",
        when(
            lit(als_max) > lit(als_min),
            (
                col("als_score") - lit(als_min)
            ) / (
                lit(als_max) - lit(als_min)
            )
        ).otherwise(lit(0.0))
    )
)

Content scores are already approximately between zero and one

In [83]:
content_candidates_normalized = (
    content_candidates_filtered
    .withColumn(
        "content_score_normalized",
        when(
            col("content_score") < 0,
            lit(0.0)
        )
        .when(
            col("content_score") > 1,
            lit(1.0)
        )
        .otherwise(col("content_score"))
    )
)

## Build Hybrid Recommendation

Use a full outer join so candidates found by either model remain available.

In [84]:
hybrid_candidates = (
    als_candidates_normalized.select(
        "user_id",
        "product_id",
        "als_score_normalized"
    )
    .join(
        content_candidates_normalized.select(
            "user_id",
            "product_id",
            "content_score_normalized"
        ),
        ["user_id", "product_id"],
        "full_outer"
    )
    .fillna({
        "als_score_normalized": 0.0,
        "content_score_normalized": 0.0
    })
)

In [85]:
# Blend Scores
ALS_WEIGHT = 0.65
CONTENT_WEIGHT = 0.35

hybrid_candidates = (
    hybrid_candidates
    .withColumn(
        "hybrid_score",
        ALS_WEIGHT * col("als_score_normalized") +
        CONTENT_WEIGHT * col("content_score_normalized")
    )
)

The collaborative component makes recommendations more personalised using similar-user behaviour. The content component preserves relevance to product attributes. This does not guarantee that every recommendation will be useful, but it greatly reduces irrelevant recommendations compared with using either method alone.

## Apply Hard Business Filters

Prevent recommendations that are clearly unsuitable

In [86]:
hybrid_candidates = (
    hybrid_candidates
    .join(
        product_dimension.select(
            "product_id",
            "product_name",
            "category_name",
            "subcategory_id",
            "base_price",
            "current_stock",
            "is_active"
        ),
        "product_id",
        "inner"
    )
    .filter(col("is_active") == True)
    .filter(col("current_stock") > 0)
    .filter(col("base_price") > 0)
)

In [87]:
# Price Guardrail
user_price_profile = (
    train_items
    .groupBy("user_id")
    .agg(
        avg("unit_price").alias("average_user_price")
    )
)

hybrid_candidates = (
    hybrid_candidates
    .join(
        user_price_profile,
        "user_id",
        "left"
    )
    .filter(
        col("base_price") <=
        (
            col("average_user_price") * 3.0 +
            lit(1.0)
        )
    )
)

This avoids recommending products far outside the user’s historical price range

## Rank The Top Recommendations

In [88]:
recommendation_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        col("hybrid_score").desc(),
        col("product_id")
    )
)

top_recommendations = (
    hybrid_candidates
    .withColumn(
        "recommendation_rank",
        row_number().over(recommendation_window)
    )
    .filter(col("recommendation_rank") <= 10)
)

In [89]:
# Display
top_recommendations.select(
    "user_id",
    "recommendation_rank",
    "product_id",
    "product_name",
    "category_name",
    "als_score_normalized",
    "content_score_normalized",
    "hybrid_score"
).show(50, truncate=False)

+-----------+-------------------+----------+-------------------------------------------------+-------------------------------+--------------------+------------------------+-------------------+
|user_id    |recommendation_rank|product_id|product_name                                     |category_name                  |als_score_normalized|content_score_normalized|hybrid_score       |
+-----------+-------------------+----------+-------------------------------------------------+-------------------------------+--------------------+------------------------+-------------------+
|user_000003|1                  |prod_02905|Managed 3Rdgeneration Definition                 |Smith Ltd                      |0.409610703233672   |0.3810594283945383      |0.39961775703997515|
|user_000003|2                  |prod_00825|Versatile Tangible Secured Line                  |Moore, Becker and Carlson      |0.3550629222515757  |0.46529785247432315     |0.3936451478295373 |
|user_000003|3                  |pr

## Evaluate Againsy The Latest Transactions

In [91]:
# Create the actual Test Products
from pyspark.sql.functions import collect_set
actual_test_products = (
    test_items
    .groupBy("user_id")
    .agg(
        collect_set("product_id").alias(
            "actual_products"
        )
    )
)

In [92]:
# Create Predicted Products
from pyspark.sql.functions import (
    collect_list,
    collect_set,
    array_intersect,
    size
)

predicted_products = (
    top_recommendations
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .groupBy("user_id")
    .agg(
        collect_list("product_id").alias(
            "predicted_products"
        )
    )
)

In [93]:
# Evaluate Recall@10 and Precision@10
evaluation = (
    actual_test_products
    .join(
        predicted_products,
        "user_id",
        "inner"
    )
    .withColumn(
        "hits",
        size(
            array_intersect(
                col("actual_products"),
                col("predicted_products")
            )
        )
    )
    .withColumn(
        "precision_at_10",
        col("hits") /
        size("predicted_products")
    )
    .withColumn(
        "recall_at_10",
        col("hits") /
        size("actual_products")
    )
)

In [94]:
# Summary
evaluation.agg(
    avg("precision_at_10").alias(
        "mean_precision_at_10"
    ),
    avg("recall_at_10").alias(
        "mean_recall_at_10"
    )
).show()

+--------------------+--------------------+
|mean_precision_at_10|   mean_recall_at_10|
+--------------------+--------------------+
|5.199999999999997E-4|0.003549999999999...|
+--------------------+--------------------+



In [95]:
# Coverage
catalog_size = product_dimension.select(
    "product_id"
).distinct().count()

recommended_catalog_size = (
    top_recommendations
    .select("product_id")
    .distinct()
    .count()
)

catalog_coverage = (
    recommended_catalog_size /
    catalog_size
)

print("Catalog coverage:", catalog_coverage)

Catalog coverage: 0.9960017769880053


In [98]:
# Save the central recommendation outputa
top_recommendations.repartition(8).write \
    .mode("overwrite") \
    .parquet(
        f"{ANALYTICS_ROOT}/hybrid_recommendations"
    )

evaluation.repartition(4).write \
    .mode("overwrite") \
    .parquet(
        f"{ANALYTICS_ROOT}/recommendation_evaluation"
    )

user_mapping.write \
    .mode("overwrite") \
    .parquet(
        f"{MODEL_ROOT}/mappings/users"
    )

product_mapping.write \
    .mode("overwrite") \
    .parquet(
        f"{MODEL_ROOT}/mappings/products"
    )